<a href="https://colab.research.google.com/github/Dzikriyah/2418048_DataCleansing/blob/main/2418048DataCleansing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1. IMPORT LIBRARIES**


In [222]:
import pandas as pd
import numpy as np
import calendar

In [223]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **2. MEMBACA DATASET KOTOR**

In [224]:
# Baca file .xlsx
file_path = '/content/drive/MyDrive/KULIAH DAN RANDOM/DATA MINING/T1/dataset_kotor_psikolog.xlsx'
data = pd.read_excel(file_path, engine='openpyxl')  # Gunakan openpyxl untuk file .xlsx

In [225]:
# Untuk Menampilkan Data
print(data)

   ID_Pasien           Nama_Pasien        Usia Jenis_Kelamin  \
0      P-001          Budi Santoso          56             P   
1      P-002           Siti Aminah          38     Perempuan   
2      P-003         Rizky Pratama          40             L   
3      P-004          Dewi Lestari         NaN        female   
4      P-004           andi wijaya          20          male   
5      P-006            Maya Putri          41     Laki-laki   
6      P-007          EKO PRASETYO          55             P   
7      P-008          Rina Gunawan      25 thn     Perempuan   
8      P-009       Fajar Hidayat            39             L   
9      P-010           nani wijaya          44        female   
10     P-011           Dian Sastro          33          male   
11    P-0011           Gita Gutawa          20     Laki-laki   
12     P-013       Hendra Setiawan          38             P   
13     P-014         INDAH PERMATA          21     Perempuan   
14     P-015           joko widodo      

### **3. MENAMPILKAN STRUKTUR DATA PASIEN**

In [226]:
print("Struktur Data Pasien:")
print(data.info())

Struktur Data Pasien:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID_Pasien             67 non-null     object 
 1   Nama_Pasien           67 non-null     object 
 2   Usia                  66 non-null     object 
 3   Jenis_Kelamin         67 non-null     object 
 4   Tanggal_Pendaftaran   66 non-null     object 
 5   Tingkat_Stres_Awal    66 non-null     object 
 6   Jumlah_Sesi           66 non-null     object 
 7   Kehadiran_Sesi_Pct    66 non-null     object 
 8   Biaya_Per_Sesi_IDR    66 non-null     object 
 9   Skor_Kecemasan_Akhir  66 non-null     float64
 10  Status_Kelulusan      66 non-null     object 
 11  Rekomendasi_Lanjutan  66 non-null     object 
dtypes: float64(1), object(11)
memory usage: 6.4+ KB
None


In [227]:
column_types = data.dtypes
print(column_types)

ID_Pasien                object
Nama_Pasien              object
Usia                     object
Jenis_Kelamin            object
Tanggal_Pendaftaran      object
Tingkat_Stres_Awal       object
Jumlah_Sesi              object
Kehadiran_Sesi_Pct       object
Biaya_Per_Sesi_IDR       object
Skor_Kecemasan_Akhir    float64
Status_Kelulusan         object
Rekomendasi_Lanjutan     object
dtype: object


4. CLEANSING ID PASIEN


In [228]:
def format_ID_Pasien(kode):
    if kode.startswith("P-"):
        angka = kode[2:]
        return "P-" + angka.zfill(3)
    return kode


# Format ID terlebih dahulu
data['ID_Pasien'] = data['ID_Pasien'].astype(str).apply(format_ID_Pasien)

# Membuat ID menjadi unik
data['ID_Pasien'] = [
    f"P-{i:03d}" for i in range(1, len(data) + 1)
]

data['ID_Pasien'].head()


5. CLEANSING NAMA

In [230]:
data['Nama_Pasien'] = (
    data['Nama_Pasien']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)
data['Nama_Pasien'].head()

,Nama_Pasien
0,Budi Santoso
1,Siti Aminah
2,Rizky Pratama
3,Dewi Lestari
4,Andi Wijaya


6. CLEANSING UMUR

In [231]:
# Bersihkan nama terlebih dahulu
data['Nama_Pasien'] = (
    data['Nama_Pasien']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

# Ubah usia menjadi angka
data['Usia'] = pd.to_numeric(data['Usia'], errors='coerce')

# Ambil usia yang tersedia berdasarkan nama
usia_pasien = data.groupby('Nama_Pasien')['Usia'].transform('first')

# Isi usia yang kosong
data['Usia'] = data['Usia'].fillna(usia_pasien)

data['Usia'].head()

,Usia
0,56.0
1,38.0
2,40.0
3,49.0
4,20.0


7. CLEANSING GENDER

In [232]:
def bersihkan_jenis_kelamin(jenis_kelamin):
    jenis_kelamin = str(jenis_kelamin).strip().lower()

    if jenis_kelamin in ['p', 'perempuan', 'female']:
        return 'P'
    elif jenis_kelamin in ['l', 'laki-laki', 'male']:
        return 'L'
    else:
        return None

data['Jenis_Kelamin'] = data['Jenis_Kelamin'].apply(bersihkan_jenis_kelamin)

data['Jenis_Kelamin'].head()

,Jenis_Kelamin
0,P
1,P
2,L
3,P
4,L


8. CLEANSING TGL

In [233]:


def bersihkan_tanggal(tanggal):
    # Jika kosong
    if pd.isna(tanggal) or str(tanggal).strip() == '':
        return pd.NaT

    tanggal = str(tanggal).strip()
    tanggal = tanggal.replace('/', '-')
    bagian = tanggal.split('-')

    try:
        # Format YYYY-MM-DD
        if len(bagian[0]) == 4:
            tahun = int(bagian[0])
            bulan = int(bagian[1])
            hari = int(bagian[2])
        else:
            # Format DD-MM-YYYY
            hari = int(bagian[0])
            bulan = int(bagian[1])
            tahun = int(bagian[2])

        # Tahun > 2026 menjadi 2026
        if tahun > 2026:
            tahun = 2026

        # Bulan > 12 menjadi 12
        if bulan > 12:
            bulan = 12

        # Bulan < 1 menjadi 1
        if bulan < 1:
            bulan = 1

        # Menentukan hari terakhir bulan
        hari_terakhir = calendar.monthrange(tahun, bulan)[1]

        # Hari > hari terakhir bulan
        if hari > hari_terakhir:
            hari = hari_terakhir

        # Hari < 1 menjadi 1
        if hari < 1:
            hari = 1

        return pd.Timestamp(tahun, bulan, hari)

    except:
        return pd.NaT


# Cleansing tanggal
data['Tanggal_Pendaftaran'] = data['Tanggal_Pendaftaran'].apply(
    bersihkan_tanggal
)

# Ambil modus tanggal
modus_tanggal = data['Tanggal_Pendaftaran'].mode()[0]

# Isi data kosong dengan modus
data['Tanggal_Pendaftaran'] = data['Tanggal_Pendaftaran'].fillna(modus_tanggal)

# Format akhir YYYY-MM-DD
data['Tanggal_Pendaftaran'] = data['Tanggal_Pendaftaran'].dt.strftime('%Y-%m-%d')

data['Tanggal_Pendaftaran'].head()

,Tanggal_Pendaftaran
0,2024-02-21
1,2024-04-08
2,2024-12-22
3,2024-07-19
4,2024-04-03


9. CLEANSING TINGKAT STRESS

In [234]:
# Mengubah data menjadi numerik
data['Tingkat_Stres_Awal'] = pd.to_numeric(
    data['Tingkat_Stres_Awal'],
    errors='coerce'
)

# Nilai di luar rentang 0-100 dianggap tidak valid
data.loc[
    (data['Tingkat_Stres_Awal'] < 0) |
    (data['Tingkat_Stres_Awal'] > 100),
    'Tingkat_Stres_Awal'
] = np.nan

# Mengambil median dari data yang valid
median_stres = data['Tingkat_Stres_Awal'].median()

# Mengisi data kosong/tidak valid dengan median
data['Tingkat_Stres_Awal'] = data['Tingkat_Stres_Awal'].fillna(median_stres)

# Mengubah menjadi bilangan bulat
data['Tingkat_Stres_Awal'] = data['Tingkat_Stres_Awal'].astype(int)

data['Tingkat_Stres_Awal'].head()

,Tingkat_Stres_Awal
0,78
1,88
2,60
3,85
4,71


TAMBAH KATEGORI TINGKAT STRESS

In [235]:
# Membuat kategori tingkat stres
def kategori_stres(nilai):
    if nilai < 40:
        return 'Rendah'
    elif nilai < 70:
        return 'Sedang'
    else:
        return 'Tinggi'

data['Kategori_Stres'] = data['Tingkat_Stres_Awal'].apply(
    kategori_stres
)

# Memindahkan Kategori_Stres tepat setelah Tingkat_Stres_Awal
kolom = data.columns.tolist()

kolom.remove('Kategori_Stres')

posisi = kolom.index('Tingkat_Stres_Awal') + 1

kolom.insert(posisi, 'Kategori_Stres')

data = data[kolom]

# Melihat hasil
data[['Tingkat_Stres_Awal', 'Kategori_Stres']].head()

,Tingkat_Stres_Awal,Kategori_Stres
0,78,Tinggi
1,88,Tinggi
2,60,Sedang
3,85,Tinggi
4,71,Tinggi


10. CLEANSING JUMLAH SESI

In [236]:
# Mengubah data menjadi string terlebih dahulu
data['Jumlah_Sesi'] = data['Jumlah_Sesi'].astype(str)

# Menghapus tulisan "sesi"
data['Jumlah_Sesi'] = data['Jumlah_Sesi'].str.replace(
    'sesi', '', case=False
).str.strip()

# Mengubah menjadi angka
data['Jumlah_Sesi'] = pd.to_numeric(
    data['Jumlah_Sesi'],
    errors='coerce'
)

# Nilai negatif dianggap tidak valid
data.loc[
    data['Jumlah_Sesi'] < 0,
    'Jumlah_Sesi'
] = np.nan

# Membulatkan jumlah sesi ke bilangan terdekat
data['Jumlah_Sesi'] = data['Jumlah_Sesi'].round()

# Mengambil median dari data yang valid
median_sesi = data['Jumlah_Sesi'].median()

# Mengisi data kosong/tidak valid dengan median
data['Jumlah_Sesi'] = data['Jumlah_Sesi'].fillna(median_sesi)

# Memastikan hasil akhir berupa bilangan bulat
data['Jumlah_Sesi'] = data['Jumlah_Sesi'].astype(int)

data['Jumlah_Sesi'].head()

,Jumlah_Sesi
0,12
1,11
2,12
3,9
4,6


11. CLEANSING KEHADIRAN

In [237]:
# Ubah menjadi string
data['Kehadiran_Sesi_Pct'] = (
    data['Kehadiran_Sesi_Pct']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.strip()
)

# Ubah menjadi angka
data['Kehadiran_Sesi_Pct'] = pd.to_numeric(
    data['Kehadiran_Sesi_Pct'],
    errors='coerce'
)

# Nilai desimal 0-1 diubah menjadi persen
data.loc[
    data['Kehadiran_Sesi_Pct'] <= 1,
    'Kehadiran_Sesi_Pct'
] = data.loc[
    data['Kehadiran_Sesi_Pct'] <= 1,
    'Kehadiran_Sesi_Pct'
] * 100

# Nilai di luar rentang 0-100 dianggap tidak valid
data.loc[
    (data['Kehadiran_Sesi_Pct'] < 0) |
    (data['Kehadiran_Sesi_Pct'] > 100),
    'Kehadiran_Sesi_Pct'
] = np.nan

# Ambil median
median_kehadiran = data['Kehadiran_Sesi_Pct'].median()

# Isi data kosong dengan median
data['Kehadiran_Sesi_Pct'] = data['Kehadiran_Sesi_Pct'].fillna(
    median_kehadiran
)

# Bulatkan menjadi bilangan bulat
data['Kehadiran_Sesi_Pct'] = data['Kehadiran_Sesi_Pct'].round().astype(int)

# Tambahkan tanda %
data['Kehadiran_Sesi_Pct'] = (
    data['Kehadiran_Sesi_Pct'].astype(str) + '%'
)

data['Kehadiran_Sesi_Pct'].head()

,Kehadiran_Sesi_Pct
0,61%
1,68%
2,84%
3,62%
4,69%


12. CLEANSING BIAYA

In [238]:
# Membersihkan format Biaya Per Sesi
data['Biaya_Per_Sesi_IDR'] = (
    data['Biaya_Per_Sesi_IDR']
    .astype(str)
    .str.strip()
    .str.replace('Rp', '', regex=False)
    .str.replace('.', '', regex=False)
)

# Mengubah nilai yang menggunakan K menjadi angka rupiah
mask_k = data['Biaya_Per_Sesi_IDR'].str.upper().str.endswith('K')

data.loc[mask_k, 'Biaya_Per_Sesi_IDR'] = (
    data.loc[mask_k, 'Biaya_Per_Sesi_IDR']
    .str.upper()
    .str.replace('K', '', regex=False)
    .astype(float) * 1000
)

# Mengubah menjadi numerik
data['Biaya_Per_Sesi_IDR'] = pd.to_numeric(
    data['Biaya_Per_Sesi_IDR'],
    errors='coerce'
)

# Mencari modus
modus_biaya = data['Biaya_Per_Sesi_IDR'].mode()[0]

# Mengisi nilai kosong dengan modus
data['Biaya_Per_Sesi_IDR'] = data['Biaya_Per_Sesi_IDR'].fillna(modus_biaya)

# Menjadikan bilangan bulat
data['Biaya_Per_Sesi_IDR'] = data['Biaya_Per_Sesi_IDR'].astype(int)

print("Modus Biaya Per Sesi:", modus_biaya)

data['Biaya_Per_Sesi_IDR'].head()

Modus Biaya Per Sesi: 150000.0


,Biaya_Per_Sesi_IDR
0,250000
1,150000
2,150000
3,150000
4,150000


13. CLEANSING SKOR KECEMASAN

In [239]:
# Ubah data menjadi numerik
data['Skor_Kecemasan_Akhir'] = pd.to_numeric(
    data['Skor_Kecemasan_Akhir'],
    errors='coerce'
)

# Nilai di luar rentang 0–100 dianggap tidak valid
data.loc[
    (data['Skor_Kecemasan_Akhir'] < 0) |
    (data['Skor_Kecemasan_Akhir'] > 100),
    'Skor_Kecemasan_Akhir'
] = np.nan

# Hitung median dari data yang valid
median_kecemasan = data['Skor_Kecemasan_Akhir'].median()

# Isi data kosong dan data tidak valid dengan median
data['Skor_Kecemasan_Akhir'] = (
    data['Skor_Kecemasan_Akhir']
    .fillna(median_kecemasan)
    .round()
    .astype(int)
)

data['Skor_Kecemasan_Akhir'].head()

,Skor_Kecemasan_Akhir
0,56
1,55
2,22
3,47
4,55


KATEGORI TINGKAT KECEMASAN

In [240]:
# Membuat kategori tingkat kecemasan
def kategori_kecemasan(nilai):
    if nilai < 40:
        return 'Rendah'
    elif nilai < 70:
        return 'Sedang'
    else:
        return 'Tinggi'

data['Kategori_Kecemasan'] = data['Skor_Kecemasan_Akhir'].apply(
    kategori_kecemasan
)

# Memindahkan Kategori_Kecemasan tepat setelah Skor_Kecemasan_Akhir
kolom = data.columns.tolist()

kolom.remove('Kategori_Kecemasan')

posisi = kolom.index('Skor_Kecemasan_Akhir') + 1

kolom.insert(posisi, 'Kategori_Kecemasan')

data = data[kolom]

# Melihat hasil
data[['Skor_Kecemasan_Akhir', 'Kategori_Kecemasan']].head()

,Skor_Kecemasan_Akhir,Kategori_Kecemasan
0,56,Sedang
1,55,Sedang
2,22,Rendah
3,47,Sedang
4,55,Sedang


14. CLEANSING STATUS KELULUSAN

In [241]:
# 1. Selaraskan penamaan Status_Kelulusan
data['Status_Kelulusan'] = (
    data['Status_Kelulusan']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({
        'selesai': 'Selesai',
        'drop out': 'Drop Out',
        'drop_out': 'Drop Out',
        'masih terapi': 'Masih Terapi'
    })
)

# 2. Pastikan Skor_Kecemasan_Akhir berbentuk angka
data['Skor_Kecemasan_Akhir'] = pd.to_numeric(
    data['Skor_Kecemasan_Akhir'],
    errors='coerce'
)

# 3. Cari median skor kecemasan dari setiap status
median_status = data.groupby(
    'Status_Kelulusan'
)['Skor_Kecemasan_Akhir'].median()

print("Median skor berdasarkan status:")
print(median_status)

# 4. Isi Status_Kelulusan yang kosong
def isi_status(row):
    if pd.isna(row['Status_Kelulusan']) or row['Status_Kelulusan'] == '':
        selisih = (
            median_status - row['Skor_Kecemasan_Akhir']
        ).abs()

        return selisih.idxmin()

    return row['Status_Kelulusan']

data['Status_Kelulusan'] = data.apply(isi_status, axis=1)

# 5. Tampilkan hasil
data['Status_Kelulusan'].head()

Median skor berdasarkan status:
Status_Kelulusan
Drop Out        45.0
Masih Terapi    46.0
Selesai         49.0
Name: Skor_Kecemasan_Akhir, dtype: float64


,Status_Kelulusan
0,Selesai
1,Drop Out
2,Drop Out
3,Masih Terapi
4,Selesai


15. CLEANSING REKOMENDASI LANJUTAN

In [242]:
# 6. Selaraskan dan konsistenkan Rekomendasi_Lanjutan
aturan_rekomendasi = {
    'Selesai': 'Pemulihan / Selesai',
    'Drop Out': 'Konseling Lanjutan',
    'Masih Terapi': 'Rujukan Psikiater'
}

# 7. Isi dan sesuaikan Rekomendasi_Lanjutan berdasarkan Status_Kelulusan
data['Rekomendasi_Lanjutan'] = data['Status_Kelulusan'].map(
    aturan_rekomendasi
)

# 8. Tampilkan hasil
data[['Status_Kelulusan', 'Rekomendasi_Lanjutan']].head()

,Status_Kelulusan,Rekomendasi_Lanjutan
0,Selesai,Pemulihan / Selesai
1,Drop Out,Konseling Lanjutan
2,Drop Out,Konseling Lanjutan
3,Masih Terapi,Rujukan Psikiater
4,Selesai,Pemulihan / Selesai


16. IMPORT HASIL DATASET CLEANSING

In [243]:
# Simpan hasil cleansing
data.to_excel(
    'data_hasil_cleansing.xlsx',
    index=False
)

# Download file Excel
from google.colab import files
files.download('data_hasil_cleansing.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>